In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.image as mpimg
import matplotlib.patches as patches
import matplotlib as mpl
from mplsoccer import Pitch, VerticalPitch, FontManager, Sbopen, add_image
from matplotlib.font_manager import FontProperties
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patheffects as path_effects
from highlight_text import ax_text, fig_text
from PIL import Image
from mplsoccer import add_image
from urllib.request import urlopen
import os


# Print the modified DataFrame
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

green = '#69f900'
red = '#ff4b44'
blue = '#00a0de'
violet = '#a369ff'
bg_color= '#f5f5f5'
line_color= '#000000'
col1 = '#ff4b44'
col2 = '#00a0de'

# # UCL
# col1 = '#f659fd'
# col2 = '#33efff'
# bg_color = '#060F38'
# line_color = '#ffffff'
# green = '#69f900'
# red = '#ff4b44'

In [4]:
df = pd.read_csv(r"D:\FData\All_Top_5_League\2024_25\event_data\Premier_League_2024_25_event_data.csv")

C:\Users\h\AppData\Local\Temp\ipykernel_10100\2731803387.py:1: DtypeWarning: Columns (33) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r"D:\FData\All_Top_5_League\2024_25\event_data\Premier_League_2024_25_event_data.csv")


In [5]:
dft = df[df['type'] == 'Goal']
dft = dft[['matchId', 'type', 'isGoal', 'qualifiers', 'minute', 'second', 'teamName', 'oppositionTeamName']]

In [6]:
dft.head()

,matchId,type,isGoal,qualifiers,minute,second,teamName,oppositionTeamName
1759,1821049.0,Goal,1.0,"[{'type': {'value': 178, 'displayName': 'Stand...",86.0,46.0,Man Utd,Fulham
3020,1821050.0,Goal,1.0,"[{'type': {'value': 154, 'displayName': 'Inten...",59.0,26.0,Liverpool,Ipswich
3104,1821050.0,Goal,1.0,"[{'type': {'value': 89, 'displayName': 'OneOnO...",64.0,48.0,Liverpool,Ipswich
3792,1821057.0,Goal,1.0,"[{'type': {'value': 103, 'displayName': 'GoalM...",3.0,29.0,Aston Villa,West Ham
4393,1821057.0,Goal,1.0,"[{'type': {'value': 17, 'displayName': 'BoxCen...",36.0,56.0,West Ham,Aston Villa


In [7]:
def swap_own_goal_teams(df):
    """
    Swap teamName and oppositionTeamName when qualifiers contains 'OwnGoal'
    
    Parameters:
    df: DataFrame with columns including 'qualifiers', 'teamName', 'oppositionTeamName'
    
    Returns:
    DataFrame with swapped team names for own goals
    """
    
    # Create a copy to avoid modifying the original dataframe
    df_modified = df.copy()
    
    # Check if qualifiers column contains 'OwnGoal' (case-insensitive)
    # Handle both string and list cases for qualifiers
    own_goal_mask = df_modified['qualifiers'].astype(str).str.contains('OwnGoal', case=False, na=False)
    
    # For rows with own goals, swap the team names
    if own_goal_mask.any():
        # Store original values
        original_team = df_modified.loc[own_goal_mask, 'teamName'].copy()
        original_opposition = df_modified.loc[own_goal_mask, 'oppositionTeamName'].copy()
        
        # Swap the values
        df_modified.loc[own_goal_mask, 'teamName'] = original_opposition
        df_modified.loc[own_goal_mask, 'oppositionTeamName'] = original_team
        
        print(f"Swapped team names for {own_goal_mask.sum()} own goal entries")
    else:
        print("No own goals found in the data")
    
    return df_modified

# Example usage:

# Method 1: Create a new dataframe with swapped values
df_corrected = swap_own_goal_teams(dft)

Swapped team names for 33 own goal entries


In [8]:
dfc = df_corrected[['matchId', 'isGoal', 'minute', 'second', 'teamName']]

In [ ]:
def calculate_comeback_wins(df):
    """
    Calculate comeback wins for each team in each match.
    
    A comeback win is when a team:
    1. Was behind at some point during the match
    2. Eventually won the match (scored more goals than opponent)
    
    Parameters:
    df: DataFrame with columns ['matchId', 'teamName', 'isGoal', 'minute', 'second']
    
    Returns:
    DataFrame with comeback wins summary
    """
    
    # Filter only goal events
    goals_df = df[df['isGoal'] == True].copy()
    
    # Create a time column for sorting (minute * 60 + second)
    goals_df['total_seconds'] = goals_df['minute'] * 60 + goals_df['second']
    
    # Sort by match, then by time
    goals_df = goals_df.sort_values(['matchId', 'total_seconds'])
    
    comeback_results = []
    
    # Process each match
    for match_id in goals_df['matchId'].unique():
        match_goals = goals_df[goals_df['matchId'] == match_id].copy()
        
        # Get unique teams in this match
        teams = match_goals['teamName'].unique()
        if len(teams) != 2:
            continue  # Skip matches that don't have exactly 2 teams
        
        team1, team2 = teams[0], teams[1]
        
        # Track running score throughout the match
        team1_score = 0
        team2_score = 0
        team1_was_behind = False
        team2_was_behind = False
        
        # Go through each goal chronologically
        for _, goal in match_goals.iterrows():
            if goal['teamName'] == team1:
                team1_score += 1
            else:
                team2_score += 1
            
            # Check if either team is behind after this goal
            if team1_score < team2_score:
                team1_was_behind = True
            elif team2_score < team1_score:
                team2_was_behind = True
        
        # Final scores
        final_team1_score = team1_score
        final_team2_score = team2_score
        
        # Determine comeback wins
        team1_comeback = False
        team2_comeback = False
        
        if final_team1_score > final_team2_score and team1_was_behind:
            team1_comeback = True
        elif final_team2_score > final_team1_score and team2_was_behind:
            team2_comeback = True
        
        # Add results
        comeback_results.append({
            'matchId': match_id,
            'teamName': team1,
            'final_score': final_team1_score,
            'opponent_score': final_team2_score,
            'won_match': final_team1_score > final_team2_score,
            'was_behind': team1_was_behind,
            'comeback_win': team1_comeback
        })
        
        comeback_results.append({
            'matchId': match_id,
            'teamName': team2,
            'final_score': final_team2_score,
            'opponent_score': final_team1_score,
            'won_match': final_team2_score > final_team1_score,
            'was_behind': team2_was_behind,
            'comeback_win': team2_comeback
        })
    
    return pd.DataFrame(comeback_results)

# Example usage:
# Assuming your dataframe is called 'df'
# comeback_df = calculate_comeback_wins(df)

# To get summary by team across all matches:
def summarize_comeback_wins(comeback_df):
    """
    Summarize comeback wins by team across all matches
    Focus on comeback ratio: how often teams come back when they fall behind
    """
    summary = comeback_df.groupby('teamName').agg({
        'comeback_win': 'sum',
        'was_behind': 'sum',
        'won_match': 'sum',
        'matchId': 'count'
    }).rename(columns={
        'comeback_win': 'total_comeback_wins',
        'was_behind': 'times_fallen_behind',
        'won_match': 'total_wins',
        'matchId': 'total_matches'
    })
    
    # Calculate comeback ratio: comebacks / times fallen behind
    summary['comeback_ratio'] = (summary['total_comeback_wins'] / 
                               summary['times_fallen_behind']).round(3)
    
    # Also calculate comeback percentage for easier interpretation
    summary['comeback_percentage'] = (summary['total_comeback_wins'] / 
                                    summary['times_fallen_behind'] * 100).round(1)
    
    # Add some additional useful metrics
    summary['times_behind_but_lost'] = summary['times_fallen_behind'] - summary['total_comeback_wins']
    
    return summary

# To get results by match:
def get_detailed_comeback_analysis(comeback_df):
    """
    Get detailed analysis of comeback performance by team
    """
    analysis = comeback_df.groupby('teamName').agg({
        'comeback_win': 'sum',
        'was_behind': 'sum',
        'won_match': 'sum',
        'matchId': 'count'
    }).rename(columns={
        'comeback_win': 'comeback_wins',
        'was_behind': 'times_behind',
        'won_match': 'total_wins',
        'matchId': 'total_matches'
    })
    
    # Key metric: How often do they come back when behind?
    analysis['comeback_success_rate'] = (analysis['comeback_wins'] / 
                                       analysis['times_behind'] * 100).round(1)
    
    # Additional insights
    analysis['times_behind_and_lost'] = analysis['times_behind'] - analysis['comeback_wins']
    analysis['resilience_score'] = analysis['comeback_wins'] / analysis['times_behind']
    analysis['percentage_matches_behind'] = (analysis['times_behind'] / 
                                           analysis['total_matches'] * 100).round(1)

# Calculate comeback wins
comeback_results = calculate_comeback_wins(dfc)

# Get the key metric: comeback ratio when teams fall behind
team_summary = summarize_comeback_wins(comeback_results)
print("Comeback performance when teams fall behind:")
team_summary[['times_fallen_behind', 'total_comeback_wins', 'comeback_ratio', 'comeback_percentage']]

# # Get detailed analysis with resilience scoring
# detailed_analysis = get_detailed_comeback_analysis(comeback_results)
# print("\nDetailed comeback analysis (sorted by success rate):")
# print(detailed_analysis)

# # Teams that are best at coming back when behind
# print("\nTop teams at coming back when behind:")
# print(detailed_analysis[['times_behind', 'comeback_wins', 'comeback_success_rate']].head())

# # Get comeback wins by match
# match_comebacks = get_comeback_wins_by_match(comeback_results)
# print("\nComeback wins by match:")
# print(match_comebacks)

Comeback performance when teams fall behind:


,times_fallen_behind,total_comeback_wins,comeback_ratio,comeback_percentage
teamName,,,,
Arsenal,8,3,0.375,37.5
Aston Villa,14,4,0.286,28.6
Bournemouth,15,3,0.200,20.0
Brentford,16,5,0.312,31.2
Brighton,17,6,0.353,35.3
Chelsea,13,4,0.308,30.8
Crystal Palace,15,2,0.133,13.3
Everton,10,2,0.200,20.0
Fulham,17,6,0.353,35.3


In [13]:
team_summary

,total_comeback_wins,times_fallen_behind,total_wins,total_matches,comeback_ratio,comeback_percentage,times_behind_but_lost
teamName,,,,,,,
Arsenal,3,8,9,22,0.375,37.5,5
Aston Villa,4,14,11,23,0.286,28.6,10
Bournemouth,3,15,8,23,0.200,20.0,12
Brentford,5,16,11,25,0.312,31.2,11
Brighton,6,17,10,27,0.353,35.3,11
Chelsea,4,13,11,22,0.308,30.8,9
Crystal Palace,2,15,5,22,0.133,13.3,13
Everton,2,10,4,16,0.200,20.0,8
Fulham,6,17,11,27,0.353,35.3,11
